# petekIO — wells, zones & reservoir quality

From raw `.wellpath` + `.las` + Petrel tops to a tidy, ordered, thickness-weighted reservoir summary — in a handful of lines. Data here is **synthetic** (generated by `synthgen`); point `GeoData` at your own export and the same calls just work.

## Load a whole field at once
Multi-bore wells, logs, and tops — auto-routed. No per-file parsing in your code.

In [1]:
import tempfile, pandas as pd
import petekio, synthgen

field = synthgen.make_field(tempfile.mkdtemp())   # synthetic 3-well field

geo = petekio.GeoData(unit='m')
for well in ['25/1-1', '25/1-2', '25/1-3']:
    geo.load_well(well, files=str(field['wells_dir']))   # .wellpath + .las, sidetracks auto-resolved
geo.load_well_tops(str(field['tops']))                    # Petrel tops -> zones

geo.well('25/1-1').bores()                                # ['', 'A', 'B']

['', 'A', 'B']

## The lithostratigraphic column — merged across every well
petekIO reads *all* wells in the tops file and merges their orderings, so a marker that pinches out in one well is placed by a well that develops it.

In [2]:
geo.strat_order

['Top Shale',
 'Top Reservoir',
 'Upper Sand',
 'Mid Sand',
 'Lower Sand B',
 'Lower Sand A',
 'Base Reservoir']

## Per-zone reservoir quality — one call, tidy and in stratigraphic order

In [3]:
w = geo.well('25/1-1')
w.zone_table('PHIE', stats=['mean', 'p50', 'p90', 'gross'], decimals=3)

,zone,bore,mean,p50,p90,gross
0,Top Shale,A,0.060,0.060,0.070,50.0
1,Top Reservoir,A,0.099,0.099,0.109,20.0
2,Upper Sand,A,0.239,0.239,0.248,30.0
3,Mid Sand,A,0.261,0.262,0.271,20.0
4,Lower Sand A,A,0.229,0.229,0.239,40.0
5,Base Reservoir,A,0.090,0.090,0.100,40.0
6,Top Shale,B,0.003,0.000,0.009,50.0
7,Top Reservoir,B,0.041,0.040,0.050,20.0
8,Upper Sand,B,0.180,0.178,0.190,30.0
9,Mid Sand,B,0.200,0.199,0.211,20.0


## Pivot to a zone × bore grid

In [4]:
w.zone_table('PHIE', pivot=True, decimals=3)

bore,A,B
zone,,
Top Shale,0.060,0.003
Top Reservoir,0.099,0.041
Upper Sand,0.239,0.180
Mid Sand,0.261,0.200
Lower Sand A,0.229,0.169
Base Reservoir,0.090,0.031


## Field roll-up: a pooled **all** row per zone, then each well/bore
`aggregate=True` adds the across-wells average first. It is **thickness-weighted** — each sample counts for the depth it represents, so a finely-sampled log can't outvote a coarse one.

In [5]:
geo.wells.zone_table('PHIE', stats=['mean', 'gross', 'samples'], aggregate=True, decimals=3)

mean  gross  samples
zone           bore                           
Top Shale      all       0.046   50.0   1099.0
               25/1-1 A  0.060   50.0    333.0
               25/1-1 B  0.003   50.0    100.0
               25/1-2    0.060   50.0    333.0
               25/1-3    0.060   50.0    333.0
Top Reservoir  all       0.085   20.0    442.0
               25/1-1 A  0.099   20.0    134.0
               25/1-1 B  0.041   20.0     40.0
               25/1-2    0.100   20.0    134.0
               25/1-3    0.099   20.0    134.0
Upper Sand     all       0.227   35.0    793.0
               25/1-1 A  0.239   30.0    200.0
               25/1-1 B  0.180   30.0     60.0
               25/1-2    0.241   30.0    200.0
               25/1-3    0.240   50.0    333.0
Mid Sand       all       0.240   20.0    306.0
               25/1-1 A  0.261   20.0    133.0
               25/1-1 B  0.200   20.0     40.0
               25/1-2    0.260   20.0    133.0
Lower Sand A   all       0.215   40.0    881.0
               25/1-1 A  0.229   40.0    267.0
               25/1-1 B  0.169   40.0     80.0
               25/1-2    0.230   40.0    267.0
               25/1-3    0.230   40.0    267.0
Base Reservoir all       0.075   40.0    878.0
               25/1-1 A  0.090   40.0    266.0
               25/1-1 B  0.031   40.0     80.0
               25/1-2    0.090   40.0    266.0
               25/1-3    0.089   40.0    266.0

## Why thickness-weighting matters
Bore B is sampled coarsely and reads lower. The plain sample mean is dominated by the densely sampled bore; the thickness-weighted mean treats the bores by the rock they cover.

In [6]:
zones = ['Upper Sand', 'Mid Sand', 'Lower Sand A']
wt = w.zone_table('PHIE', zones=zones, aggregate=True)['mean'].xs('all', level='bore')
pl = w.zone_table('PHIE', zones=zones, aggregate=True, weighted=False)['mean'].xs('all', level='bore')
pd.DataFrame({'thickness-weighted': wt, 'plain mean': pl}).round(3)

,thickness-weighted,plain mean
zone,,
Upper Sand,0.210,0.226
Mid Sand,0.231,0.247
Lower Sand A,0.199,0.216


## Focus on just the zones you care about

In [7]:
w.zone_table('PHIE', zones=['Upper Sand', 'Mid Sand'], stats=['mean','p10','p50','p90'], decimals=3)

,zone,bore,mean,p10,p50,p90
0,Upper Sand,A,0.239,0.230,0.239,0.248
1,Mid Sand,A,0.261,0.252,0.262,0.271
2,Upper Sand,B,0.180,0.170,0.178,0.190
3,Mid Sand,B,0.200,0.190,0.199,0.211


## A coincident pair the data can't order — pin it with one line
`Lower Sand A`/`B` are stacked and coincident in every well, so their order is a tiebreak. A hint sets it (a real MD relationship would always win over a hint).

In [8]:
geo2 = petekio.GeoData(unit='m')
geo2.strat_hint('Lower Sand A < Lower Sand B')      # A above B
for well in ['25/1-1', '25/1-2', '25/1-3']:
    geo2.load_well(well, files=str(field['wells_dir']))
geo2.load_well_tops(str(field['tops']))
geo2.strat_order

['Top Shale',
 'Top Reservoir',
 'Upper Sand',
 'Mid Sand',
 'Lower Sand A',
 'Lower Sand B',
 'Base Reservoir']